# Cross-Validation

Runs cross-validation with optuna tuned HPs for all 21 **model x representation** combinations

In [9]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path('/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/PBSG')))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
df = pd.read_csv('data_classifier.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [11]:
with open("random_stratified_splits_cls.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=546  test=97  folds=5
Device: cpu


In [13]:
X_fp_RU        = np.load("Vectors/X_fp_RU.npy")
X_fp_pooled_RU    = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly      = np.load("Vectors/X_fp_poly.npy")
X_fp_pooled_poly = np.load("Vectors/X_fp_pooled_poly.npy")

In [14]:
graphs_PBSG = torch.load("Graphs/graphs_PBSG.pt")
graphs_SMILES = torch.load("Graphs/graphs_SMILES.pt")
graphs_SMILESplusglobal = torch.load("Graphs/graphs_SMILES+global.pt")

In [15]:
y=np.load("Vectors/y.npy")

In [17]:
from scripts.CV import run_all_gnn_cv, run_all_classical_cv, print_cv_summary

# Define your reps
graph_reps = {
    "PBSG":         graphs_PBSG,
    "SMILES":       graphs_SMILES,
    "SMILES_gl": graphs_SMILESplusglobal,
}

vector_reps = {
    "FP RU":          X_fp_RU,
    "FP+pooled RU":   X_fp_pooled_RU,
    "FP poly":        X_fp_poly,
    "FP+pooled poly": X_fp_pooled_poly,
}

In [18]:
from scripts.CV import (run_all_gnn_cv, run_all_classical_cv,
                    print_cv_summary, load_tuned_hp_cls)

hp = load_tuned_hp_cls("best_hp_full.json")

In [19]:
gnn_results = run_all_gnn_cv(graph_reps, folds, df, device, hp_dict=hp)
cls_results = run_all_classical_cv(vector_reps, y, folds, df, hp_dict=hp)

  Using tuned HPs for gcn_pbsg

  GNN CV: GCN (PBSG)  seed=0  in_ch=85  edge_dim=1  meta_dim=13
    Epoch  20 | train_loss=0.3580 val_loss=0.4047 val_auroc=0.8855 val_auprc=0.9334
    Epoch  40 | train_loss=0.2813 val_loss=0.3878 val_auroc=0.9183 val_auprc=0.9530
    Epoch  60 | train_loss=0.2576 val_loss=0.3344 val_auroc=0.9318 val_auprc=0.9566
    Epoch  80 | train_loss=0.2608 val_loss=0.4235 val_auroc=0.8869 val_auprc=0.9082
    Epoch 100 | train_loss=0.2452 val_loss=0.5986 val_auroc=0.8940 val_auprc=0.9389
    Epoch 120 | train_loss=0.2135 val_loss=0.3355 val_auroc=0.9332 val_auprc=0.9578
    Early stop at epoch 134  best_auroc=0.9491
  Fold 1: acc=0.9091  f1=0.9007  auroc=0.9491  auprc=0.9681  [40s]
    Epoch  20 | train_loss=0.3291 val_loss=0.7002 val_auroc=0.8223 val_auprc=0.8769
    Epoch  40 | train_loss=0.2677 val_loss=0.4144 val_auroc=0.8915 val_auprc=0.9195
    Epoch  60 | train_loss=0.2407 val_loss=0.5133 val_auroc=0.8711 val_auprc=0.9002
    Early stop at epoch 63  best_a

/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 2: acc=0.8716  f1=0.8527  auroc=0.9105  auprc=0.9416  [1s]
  Fold 3: acc=0.8624  f1=0.8511  auroc=0.9005  auprc=0.9400  [1s]


/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 4: acc=0.8991  f1=0.8867  auroc=0.9155  auprc=0.9421  [1s]


/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 5: acc=0.8716  f1=0.8586  auroc=0.9317  auprc=0.9584  [1s]
  LR (FP+pooled RU) mean CV: acc=0.8809  f1=0.8678  auroc=0.9204  auprc=0.9490  [total 5s]
  Using tuned HPs for rf_fp_pooled_ru

── Classical CV: RF (FP+pooled RU) ──────────────────────────────────
  Fold 1: acc=0.9091  f1=0.8982  auroc=0.9357  auprc=0.9585  [0s]
  Fold 2: acc=0.8807  f1=0.8642  auroc=0.9349  auprc=0.9611  [0s]
  Fold 3: acc=0.8899  f1=0.8788  auroc=0.9382  auprc=0.9540  [0s]
  Fold 4: acc=0.8899  f1=0.8756  auroc=0.9496  auprc=0.9716  [0s]
  Fold 5: acc=0.8899  f1=0.8773  auroc=0.9346  auprc=0.9635  [0s]
  RF (FP+pooled RU) mean CV: acc=0.8919  f1=0.8788  auroc=0.9386  auprc=0.9617  [total 2s]
  Using tuned HPs for xgb_fp_pooled_ru

── Classical CV: XGB (FP+pooled RU) ──────────────────────────────────
  Fold 1: acc=0.8545  f1=0.8348  auroc=0.9328  auprc=0.9607  [0s]
  Fold 2: acc=0.8807  f1=0.8679  auroc=0.9385  auprc=0.9623  [0s]
  Fold 3: acc=0.9083  f1=0.8963  auroc=0.9520  auprc=0.9616  [0s]
  Fo

In [20]:
print_cv_summary(gnn_results + cls_results)


  CV SUMMARY  (sorted by AUPRC)
Name                          Acc               F1                AUROC             AUPRC             Time    
--------------------------------------------------------------------------------------------------------------
XGB (FP+pooled RU)            0.8847 ± 0.0210   0.8714 ± 0.0236   0.9412 ± 0.0091   0.9645 ± 0.0052   0s      
XGB (FP+pooled poly)          0.8865 ± 0.0229   0.8706 ± 0.0288   0.9392 ± 0.0084   0.9626 ± 0.0043   2s      
RF (FP+pooled RU)             0.8919 ± 0.0093   0.8788 ± 0.0110   0.9386 ± 0.0056   0.9617 ± 0.0058   2s      
GATV2 (SMILES_gl)             0.8846 ± 0.0124   0.8711 ± 0.0131   0.9335 ± 0.0046   0.9616 ± 0.0029   60.2m   
GCN (PBSG)                    0.8937 ± 0.0199   0.8839 ± 0.0220   0.9352 ± 0.0088   0.9610 ± 0.0059   2.6m    
RF (FP+pooled poly)           0.8919 ± 0.0160   0.8797 ± 0.0184   0.9381 ± 0.0092   0.9608 ± 0.0059   1s      
GATV2 (PBSG)                  0.8937 ± 0.0238   0.8818 ± 0.0280   0.9376 ± 0.01

In [21]:
import pandas as pd

def results_to_df(results):
    rows = []
    for r in results:
        row = {
            "name":   r["name"],
            "acc":    r["mean"]["acc"],
            "acc_std": r["std"]["acc"],
            "f1":     r["mean"]["f1"],
            "f1_std": r["std"]["f1"],
            "auroc":  r["mean"]["auroc"],
            "auroc_std": r["std"]["auroc"],
            "auprc":  r["mean"]["auprc"],
            "auprc_std": r["std"]["auprc"],
            "time_s": r.get("time_s", 0),
        }
        rows.append(row)
    return pd.DataFrame(rows)

# Convert and save
all_results = gnn_results + cls_results
df_results = results_to_df(all_results)
df_results.to_csv("cv_results_full.csv", index=False)
print(df_results.sort_values("auprc", ascending=False).to_string(index=False))

                name      acc  acc_std       f1   f1_std    auroc  auroc_std    auprc  auprc_std      time_s
  XGB (FP+pooled RU) 0.884671 0.021012 0.871398 0.023606 0.941181   0.009113 0.964534   0.005249    0.498609
XGB (FP+pooled poly) 0.886455 0.022869 0.870582 0.028824 0.939184   0.008406 0.962604   0.004316    2.148985
   RF (FP+pooled RU) 0.891910 0.009296 0.878808 0.010962 0.938600   0.005642 0.961734   0.005824    1.700816
   GATV2 (SMILES_gl) 0.884621 0.012388 0.871105 0.013085 0.933521   0.004551 0.961565   0.002861 3613.734403
          GCN (PBSG) 0.893745 0.019888 0.883899 0.021990 0.935156   0.008828 0.961036   0.005903  158.714192
 RF (FP+pooled poly) 0.891910 0.015961 0.879749 0.018355 0.938119   0.009161 0.960777   0.005859    1.296907
        GATV2 (PBSG) 0.893728 0.023793 0.881773 0.028026 0.937621   0.010151 0.959945   0.005810  164.546371
         GINE (PBSG) 0.888224 0.019817 0.874811 0.022429 0.934826   0.018488 0.959784   0.011242   88.230008
     GCN (SMILES_gl

In [22]:
def fold_results_to_df(results):
    rows = []
    for r in results:
        for fold_i, f in enumerate(r["folds"]):
            rows.append({
                "name":  r["name"],
                "fold":  fold_i + 1,
                "acc":   f["acc"],
                "f1":    f["f1"],
                "auroc": f["auroc"],
                "auprc": f["auprc"],
            })
    return pd.DataFrame(rows)

df_folds = fold_results_to_df(all_results)
df_folds.to_csv("cv_results_folds.csv", index=False)